In [2]:
import json

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig
from langchain_openai import ChatOpenAI

from langgraph.graph import END, START, StateGraph, MessagesState
from langgraph.store.memory import BaseStore, InMemoryStore

import os
from dotenv import load_dotenv

load_dotenv()

# 假设 'fetch_product_recommendations'、'format_recommendation_message'、'UserProfile' 已在其他地方定义

recommendation_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个乐于助人的推荐引擎。根据用户资料，提供个性化的产品推荐。"),
    ("human", "{user_profile_summary}")
])

recommendation_chain = recommendation_prompt | ChatOpenAI(
        model=os.getenv("MODEL_NAME"),
        base_url=os.getenv("BASE_URL"),
        api_key=os.getenv("OPENAI_API_KEY"),
        temperature=0,
    ) |  (lambda x: {"messages": [AIMessage(content=x.content)]})

def recommend_products(state: MessagesState, config: RunnableConfig, store: BaseStore):
    """根据用户存储的偏好向用户推荐产品。"""
    user_id = config["configurable"]["user_id"]
    namespace = ("user_profiles", user_id)
    user_profile_record = store.get(namespace, "profile")
    user_profile = user_profile_record.value if user_profile_record else {}

    user_profile_summary = format_user_profile_summary(user_profile) # 用于格式化提示的用户资料字典的函数

    # 调用推荐链
    result = recommendation_chain.invoke({"user_profile_summary": user_profile_summary})
    return result

def format_user_profile_summary(user_profile: dict) -> str:
    """将用户资料字典格式化为字符串以进行提示注入。"""
    name = user_profile.get("preferred_name", "用户")
    categories = ", ".join(user_profile.get("preferred_product_categories", ["产品"]))
    return f"用户名为 {name}。他们偏好的产品类别是：{categories}。"


def extract_preference_updates(state: MessagesState) -> dict:
    """从最新的用户消息中提取用户偏好更新。"""
    latest_message_content = state["messages"][-2].content
    # 示例：使用 LLM 提取偏好 - 替换为实际的提取逻辑
    extraction_prompt = ChatPromptTemplate.from_messages([
        ("system", "从用户消息中提取用户的产品类别偏好。以 JSON 字典形式返回，外层不要包裹 ```json```， 键为 'preferred_product_categories'，值为类别列表。如果没有表达偏好，则返回一个空字典。"),
        ("human", "{user_message}")
    ])
    extraction_chain = extraction_prompt | ChatOpenAI(
        model=os.getenv("MODEL_NAME"),
        base_url=os.getenv("BASE_URL"),
        api_key=os.getenv("OPENAI_API_KEY"),
        temperature=0,
    ) # 如果需要结构化输出，请替换为合适的链

    preferences_json = extraction_chain.invoke({"user_message": latest_message_content})
    try:
        preferences = json.loads(preferences_json.content) # 假设 LLM 返回 JSON 字符串
        return preferences
    except json.JSONDecodeError:
        return {} # 如果提取失败，则返回空字典

def update_user_profile_node(state: MessagesState, config: RunnableConfig, store: BaseStore):
    """在"热路径"中触发的记忆存储中更新用户资料。"""
    user_id = config["configurable"]["user_id"]
    namespace = ("user_profiles", user_id)
    user_profile_record = store.get(namespace, "profile")
    user_profile = user_profile_record.value if user_profile_record else {}

    preference_updates = extract_preference_updates(state) # 从当前轮次提取偏好
    
    updated_profile = user_profile.copy() # 创建副本以避免修改原始字典
    if "preferred_product_categories" in preference_updates: # 合并或更新偏好
        updated_profile["preferred_product_categories"] = list(set(updated_profile.get("preferred_product_categories", []) + preference_updates["preferred_product_categories"])) # 示例：合并列表
    
    store.put(namespace, "profile", updated_profile) # 保存更新后的资料

    return {} # 节点应返回字典


# LangGraph 中的示例用法
memory_store = InMemoryStore()

builder = StateGraph(MessagesState)
builder.add_node("recommend_products", recommend_products)
builder.add_node("update_profile", update_user_profile_node) # 在"热路径"中更新资料的节点
builder.add_edge(START, "recommend_products")
builder.add_edge("recommend_products", "update_profile") # 在推荐后更新资料
builder.add_edge("update_profile", END)

graph = builder.compile(store=memory_store)

# 初始化用户资料
user_id = "user_123"
memory_store.put(
    ("user_profiles", user_id),
    "profile",
    {
        "preferred_name": "张三",
        "preferred_product_categories": ["电子产品", "书籍"]
    }
)

# 执行图 - 第一次交互（获取推荐）
config = {"configurable": {"user_id": user_id}}
result = graph.invoke({"messages": [HumanMessage(content="你好")]}, config=config)
print("初始推荐:")
print(result["messages"][-1].content)
print("=" * 50)

# 模拟用户表达新的偏好
user_message = "我最近对户外装备和运动鞋很感兴趣。"
result = graph.invoke(
    {"messages": result["messages"] + [HumanMessage(content=user_message)]},
    config=config
)

# 检查更新后的用户资料
updated_profile = memory_store.get(("user_profiles", user_id), "profile").value
print("更新后的用户资料:")
print(json.dumps(updated_profile, ensure_ascii=False, indent=2))
print("=" * 50)

# 再次获取推荐，应该包含新的偏好
result = graph.invoke({"messages": [HumanMessage(content="我又来了")]}, config=config)
print("基于更新后资料的推荐:")
print(result["messages"][-1].content)

初始推荐:
你好，张三！根据你感兴趣的产品类别（电子产品和书籍），我为你准备了一些推荐：

### 🎧 电子产品推荐
1. **索尼 WF-1000XM5 无线耳机**  
   - 主动降噪功能，音质出色，适合通勤和办公使用。
2. **Kindle Paperwhite 电子书阅读器**  
   - 阅读体验接近纸质书，防水设计，适合随时随地阅读。
3. **小米智能手表 S3**  
   - 可更换表圈设计，健康监测功能齐全，适合喜欢运动的你。
4. **华为 MatePad Pro 12.6 平板电脑**  
   - 大屏幕，高性能，适合阅读电子书和处理工作。
5. **Anker 便携式充电宝（20000mAh）**  
   - 大容量，支持快充，出门旅行必备。

### 📚 书籍推荐
1. **《三体》 - 刘慈欣**  
   - 中国科幻巨作，情节宏大，适合喜欢思考的你。
2. **《活着》 - 余华**  
   - 一部感人至深的小说，讲述生命的意义。
3. **《原则》 - 瑞·达利欧**  
   - 桥水基金创始人的人生和工作原则，适合对自我提升感兴趣的你。
4. **《人类简史》 - 尤瓦尔·赫拉利**  
   - 从宏观角度讲述人类历史的发展，内容深刻。
5. **《你当像鸟飞往你的山》 - 塔拉·韦斯特弗**  
   - 一本回忆录，讲述了教育如何改变人生。

如果你有更具体的需求，比如预算、用途等，可以告诉我，我会为你进一步筛选推荐！😊
更新后的用户资料:
{
  "preferred_name": "张三",
  "preferred_product_categories": [
    "运动鞋",
    "电子产品",
    "书籍",
    "户外装备"
  ]
}
基于更新后资料的推荐:
嗨，张三！

根据您偏好的产品类别，我为您整理了一些精选推荐，希望您会喜欢：

👟 运动鞋推荐：
1. Nike Air Zoom Pegasus 40 - 跑步训练两相宜，缓震性能出色
2. Adidas Ultraboost Light - 经典爆米花鞋底，潮流与舒适兼备
3. Lining 䨻系列跑鞋 - 国货之光，轻量缓震黑科技

🎧 电子产品推荐：
1. Sony WH-1000XM5 无线降噪耳机 - 顶级降

In [3]:
import json
from typing import Dict, Any, List
from langgraph.graph import StateGraph, END, START
from langgraph.store.memory import InMemoryStore, BaseStore
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# 定义任务状态类型
class TaskState(dict):
    """任务状态管理"""
    def __init__(self, **kwargs):
        super().__init__(kwargs)
        if 'messages' not in self:
            self['messages'] = []
        if 'task_data' not in self:
            self['task_data'] = {}
        if 'current_step' not in self:
            self['current_step'] = 0
        if 'task_type' not in self:
            self['task_type'] = None
        if 'task_complete' not in self:
            self['task_complete'] = False

class TaskCompletionAgent:
    """多步骤任务完成智能体"""
    
    def __init__(self):
        # 定义不同类型的任务模板
        self.task_templates = {
            "flight_booking": {
                "steps": [
                    {"field": "departure_city", "prompt": "请告诉我您的出发城市：", "required": True},
                    {"field": "arrival_city", "prompt": "请告诉我您的目的地城市：", "required": True},
                    {"field": "departure_date", "prompt": "请提供出发日期（格式：YYYY-MM-DD）：", "required": True},
                    {"field": "passengers", "prompt": "请告诉我乘客人数：", "required": True}
                ],
                "completion_message": "✈️ 航班信息已收集完毕！正在为您查找最佳航班..."
            },
            "restaurant_reservation": {
                "steps": [
                    {"field": "cuisine_type", "prompt": "您想预订哪种类型的餐厅？（如：中餐、西餐、日料等）", "required": True},
                    {"field": "date_time", "prompt": "请提供用餐日期和时间：", "required": True},
                    {"field": "party_size", "prompt": "请告诉我用餐人数：", "required": True}
                ],
                "completion_message": "🍽️ 餐厅预订信息已收集完毕！正在为您查找合适的餐厅..."
            }
        }

def process_task(state: TaskState, config: RunnableConfig, store: BaseStore):
    """处理任务的主要逻辑"""
    print(f"🔄 处理任务 - 当前状态: step={state.get('current_step', 0)}, type={state.get('task_type')}")
    
    agent = TaskCompletionAgent()
    
    # 如果还没有任务类型，先检测任务类型
    if not state.get('task_type'):
        if not state['messages']:
            return state
        
        last_message = state['messages'][-1].content.lower()
        
        # 检测任务类型
        if any(keyword in last_message for keyword in ['机票', '航班', '飞机', '预订机票']):
            task_type = "flight_booking"
        elif any(keyword in last_message for keyword in ['餐厅', '预订', '吃饭', '订餐']):
            task_type = "restaurant_reservation"
        else:
            task_type = "flight_booking"
        
        print(f"📋 检测到任务类型: {task_type}")
        
        # 初始化任务状态
        state['task_type'] = task_type
        state['current_step'] = 0
        state['task_data'] = {}
        state['task_complete'] = False
        
        # 添加确认消息和第一个问题
        template = agent.task_templates[task_type]
        first_step = template['steps'][0]
        
        confirmation_messages = {
            "flight_booking": f"我来帮您预订航班。{first_step['prompt']}",
            "restaurant_reservation": f"我来帮您预订餐厅。{first_step['prompt']}"
        }
        
        ai_message = AIMessage(content=confirmation_messages[task_type])
        state['messages'].append(ai_message)
        return state
    
    # 如果已经完成任务，直接返回
    if state.get('task_complete'):
        return state
    
    # 处理信息收集
    task_type = state['task_type']
    template = agent.task_templates[task_type]
    steps = template['steps']
    current_step = state['current_step']
    
    # 检查是否还有步骤需要完成
    if current_step >= len(steps):
        # 所有信息已收集完毕
        state['task_complete'] = True
        response = template['completion_message']
        response += "\n\n📋 收集到的信息：\n"
        for key, value in state['task_data'].items():
            response += f"• {key}: {value}\n"
        
        ai_message = AIMessage(content=response)
        state['messages'].append(ai_message)
        return state
    
    # 检查是否有新的用户消息需要处理
    if len(state['messages']) >= 2 and isinstance(state['messages'][-1], HumanMessage):
        # 获取当前步骤信息
        step_info = steps[current_step]
        field_name = step_info['field']
        user_message = state['messages'][-1].content
        
        print(f"🎯 处理用户输入: {user_message} -> 提取字段: {field_name}")
        
        # 尝试提取信息
        extracted_value = extract_field_value(user_message, field_name)
        
        if extracted_value and extracted_value != "NOT_FOUND":
            # 成功提取信息
            state['task_data'][field_name] = extracted_value
            state['current_step'] += 1
            
            print(f"✅ 成功提取 {field_name}: {extracted_value}")
            
            # 检查是否还有更多步骤
            if state['current_step'] < len(steps):
                next_step = steps[state['current_step']]
                next_prompt = next_step['prompt']
                response = f"已记录您的{field_name}：{extracted_value}。{next_prompt}"
            else:
                # 所有信息已收集完毕
                state['task_complete'] = True
                response = template['completion_message']
                response += "\n\n📋 收集到的信息：\n"
                for key, value in state['task_data'].items():
                    response += f"• {key}: {value}\n"
        else:
            # 提取失败，重新询问
            response = f"抱歉，我没能理解您提供的{field_name}信息。{step_info['prompt']}"
        
        # 添加AI响应
        ai_message = AIMessage(content=response)
        state['messages'].append(ai_message)
    
    return state

def extract_field_value(user_message: str, field_name: str) -> str:
    """简化的信息提取函数"""
    message_lower = user_message.lower()
    
    if field_name == "departure_city":
        cities = ["北京", "上海", "广州", "深圳", "杭州", "成都", "重庆", "西安", "南京", "武汉"]
        for city in cities:
            if city in user_message:
                return city
    
    elif field_name == "arrival_city":
        cities = ["北京", "上海", "广州", "深圳", "杭州", "成都", "重庆", "西安", "南京", "武汉", "大连", "青岛"]
        for city in cities:
            if city in user_message:
                return city
    
    elif field_name == "departure_date":
        import re
        date_patterns = [r'\d{4}-\d{2}-\d{2}', r'\d{1,2}月\d{1,2}日', r'\d{1,2}/\d{1,2}']
        for pattern in date_patterns:
            match = re.search(pattern, user_message)
            if match:
                return match.group()
    
    elif field_name == "passengers":
        import re
        number_match = re.search(r'(\d+)人?', user_message)
        if number_match:
            return number_match.group(1)
    
    elif field_name == "cuisine_type":
        cuisine_types = ["中餐", "西餐", "日料", "韩料", "意大利", "法餐", "泰餐", "印度"]
        for cuisine in cuisine_types:
            if cuisine in user_message:
                return cuisine
    
    elif field_name == "party_size":
        import re
        number_match = re.search(r'(\d+)人?', user_message)
        if number_match:
            return number_match.group(1)
    
    elif field_name == "date_time":
        if any(word in message_lower for word in ["明天", "今天", "后天"]):
            return user_message
        import re
        if re.search(r'\d+点|\d+:\d+', user_message):
            return user_message
    
    return "NOT_FOUND"

def save_task_memory(state: TaskState, config: RunnableConfig, store: BaseStore):
    """保存任务完成记录到长期记忆"""
    print("💾 保存任务记录到记忆...")
    
    if not state['task_data'] or not state['task_type']:
        return state
    
    user_id = config.get("configurable", {}).get("user_id", "default_user")
    
    # 创建任务记录
    task_record = {
        "task_type": state['task_type'],
        "task_data": state['task_data'],
        "completion_time": "2024-02-15T10:30:00Z",
        "status": "completed"
    }
    
    # 保存到记忆存储
    import uuid
    record_key = str(uuid.uuid4())
    namespace = (user_id, "completed_tasks")
    
    store.put(namespace, record_key, task_record)
    print(f"✅ 任务记录已保存: {record_key}")
    
    # 添加确认消息
    confirmation_message = AIMessage(content="✨ 任务已完成并保存到您的记录中！如需查看历史记录或开始新任务，请随时告诉我。")
    state['messages'].append(confirmation_message)
    
    return state

# 创建简化的任务完成图
def create_task_completion_graph():
    """创建任务完成流程图"""
    store = InMemoryStore()
    
    # 创建状态图
    workflow = StateGraph(dict)
    
    # 添加节点
    workflow.add_node("process_task", process_task)
    workflow.add_node("save_memory", save_task_memory)
    
    # 条件函数：检查是否需要保存记忆
    def should_save_memory(state):
        if state.get('task_complete', False) and state.get('task_data'):
            return "save_memory"
        else:
            return "end"
    
    # 添加边
    workflow.add_edge(START, "process_task")
    workflow.add_conditional_edges(
        "process_task",
        should_save_memory,
        {
            "save_memory": "save_memory",
            "end": END
        }
    )
    workflow.add_edge("save_memory", END)
    
    # 编译图
    app = workflow.compile(store=store)
    return app, store

# 演示任务完成系统
print("=== 情境化任务完成系统演示 ===")

# 创建应用和存储
app, store = create_task_completion_graph()

# 配置
config = {"configurable": {"user_id": "user_456"}}

print("\n--- 航班预订演示 ---")

# 演示完整的多轮对话
conversation_steps = [
    "我想预订一张机票",
    "我从北京出发", 
    "我要去上海",
    "2024-03-15",
    "1人"
]

current_state = TaskState()

for i, user_input in enumerate(conversation_steps):
    print(f"\n👤 用户: {user_input}")
    
    # 添加用户消息
    current_state['messages'].append(HumanMessage(content=user_input))
    
    # 处理任务
    result = app.invoke(current_state, config=config)
    
    # 更新状态
    current_state = result
    
    # 显示AI响应
    if current_state['messages'] and isinstance(current_state['messages'][-1], AIMessage):
        print(f"🤖 助手: {current_state['messages'][-1].content}")
    
    # 显示当前任务状态
    print(f"📊 当前步骤: {current_state.get('current_step', 0)}")
    print(f"🏁 任务完成: {current_state.get('task_complete', False)}")
    if current_state.get('task_data'):
        print(f"📋 已收集数据: {current_state['task_data']}")

# 检查保存的记忆
print("\n--- 检查保存的任务记忆 ---")
saved_memories = store.search(("user_456", "completed_tasks"))
print(f"📋 用户完成的任务数量: {len(saved_memories)}")

for memory in saved_memories:
    task_data = memory.value
    print(f"✅ {task_data['task_type']}: {task_data['task_data']}")

=== 情境化任务完成系统演示 ===

--- 航班预订演示 ---

👤 用户: 我想预订一张机票
🔄 处理任务 - 当前状态: step=0, type=None
📋 检测到任务类型: flight_booking
🤖 助手: 我来帮您预订航班。请告诉我您的出发城市：
📊 当前步骤: 0
🏁 任务完成: False

👤 用户: 我从北京出发
🔄 处理任务 - 当前状态: step=0, type=flight_booking
🎯 处理用户输入: 我从北京出发 -> 提取字段: departure_city
✅ 成功提取 departure_city: 北京
🤖 助手: 已记录您的departure_city：北京。请告诉我您的目的地城市：
📊 当前步骤: 1
🏁 任务完成: False
📋 已收集数据: {'departure_city': '北京'}

👤 用户: 我要去上海
🔄 处理任务 - 当前状态: step=1, type=flight_booking
🎯 处理用户输入: 我要去上海 -> 提取字段: arrival_city
✅ 成功提取 arrival_city: 上海
🤖 助手: 已记录您的arrival_city：上海。请提供出发日期（格式：YYYY-MM-DD）：
📊 当前步骤: 2
🏁 任务完成: False
📋 已收集数据: {'departure_city': '北京', 'arrival_city': '上海'}

👤 用户: 2024-03-15
🔄 处理任务 - 当前状态: step=2, type=flight_booking
🎯 处理用户输入: 2024-03-15 -> 提取字段: departure_date
✅ 成功提取 departure_date: 2024-03-15
🤖 助手: 已记录您的departure_date：2024-03-15。请告诉我乘客人数：
📊 当前步骤: 3
🏁 任务完成: False
📋 已收集数据: {'departure_city': '北京', 'arrival_city': '上海', 'departure_date': '2024-03-15'}

👤 用户: 1人
🔄 处理任务 - 当前状态: step=3, type=flight_booking
🎯 处理用户输入: 1人 